<a href="https://colab.research.google.com/github/Gianbattistabsn/FAIML-RL-26/blob/alessandro-PPO-SAC/part2/clone_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import subprocess

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

REPO_URL = "https://github.com/Gianbattistabsn/FAIML-RL-26.git"
REPO_BRANCH = "alessandro-PPO-SAC"

REPO_ROOT = "/content/FAIML-RL-26"
VENV = "/content/rl_env"

# ------------------------------------------------------------
# Mount Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Clone Repo
# ------------------------------------------------------------

if not os.path.exists(REPO_ROOT):
    !git clone --branch $REPO_BRANCH $REPO_URL $REPO_ROOT

# ------------------------------------------------------------
# Install venv support
# ------------------------------------------------------------

!apt-get update -qq
!apt-get install -y python3-venv

# ------------------------------------------------------------
# Recreate clean venv
# ------------------------------------------------------------

if os.path.exists(VENV):
    !rm -rf $VENV

!python -m venv $VENV

PYTHON = f"{VENV}/bin/python"
PIP = f"{VENV}/bin/pip"

# ------------------------------------------------------------
# Upgrade pip tools
# ------------------------------------------------------------

!{PIP} install -U pip setuptools wheel

# ------------------------------------------------------------
# Install stable compatible stack
# ------------------------------------------------------------

!{PIP} install \
    numpy==1.26.4 \
    gymnasium==0.29.1 \
    stable-baselines3==2.3.2 \
    pybullet \
    tensorboard \
    pyvirtualdisplay \
    moviepy \
    imageio \
    imageio-ffmpeg \
    shimmy \
    opencv-python-headless==4.9.0.80

# ------------------------------------------------------------
# Install LOCAL panda-gym from repo
# ------------------------------------------------------------
!{PIP} install -e /content/FAIML-RL-26/part2/panda-gym

print("\nSETUP COMPLETE")
print("Python executable:")
print(PYTHON)

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0


You can safely remove it manually.


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/xpu
   ---------------------------------------- 0.0/724.8 MB ? eta -:--:--
   ---------------------------------------- 4.7/724.8 MB 31.1 MB/s eta 0:00:24
    --------------------------------------- 16.8/724.8 MB 45.5 MB/s eta 0:00:16
   - -------------------------------------- 34.6/724.8 MB 58.6 MB/s eta 0:00:12
   -- ------------------------------------- 50.6/724.8 MB 62.8 MB/s eta 0:00:11
   --- ------------------------------------ 66.8/724.8 MB 65.4 MB/s eta 0:00:11
   ---- ----------------------------------- 81.3/724.8 MB 65.4 MB/s eta 0:00:10
   ---- ----------------------------------- 87.0/724.8 MB 64.0 MB/s eta 0:00:10
   ----- --------------------------------- 106.7/724.8 MB 64.1 MB/s eta 0:00:10
   ------ -------------------------------- 116.4/724.8 MB 62.9 MB/s eta 0:00:10
   ------- ------------------------------- 133.2/724.8 MB 63.3 MB/s eta 0:0

SyntaxError: invalid syntax (3561207504.py, line 71)

In [ ]:

PYTHON = "/content/rl_env/bin/python"

test_code = """
import numpy as np
import gymnasium
import pybullet
import panda_gym
import torch

print("NumPy:", np.__version__)
print("Gymnasium:", gymnasium.__version__)
print("Torch:", torch.__version__)

print("Everything OK")
"""

with open("/content/test_env.py", "w") as f:
    f.write(test_code)

!{PYTHON} /content/test_env.py



In [ ]:

PIP = "/content/rl_env/bin/pip"
REQ = "/content/FAIML-RL-26/requirements.txt"

!cat $REQ



In [ ]:

! /content/rl_env/bin/pip install tqdm rich


In [ ]:

PYTHON = "/content/rl_env/bin/python"
SCRIPT = "/content/FAIML-RL-26/part2/train_sb3.py"

ENV_TYPE = "source"
SAMPLING = "none"
TIMESTEPS = 500000

!MPLBACKEND=Agg {PYTHON} {SCRIPT} \
    --env-type {ENV_TYPE} \
    --sampling-strategy {SAMPLING} \
    --timesteps {TIMESTEPS}

## Training Configuration

Edit the variables below, then run the Training cell.


In [5]:

# ── Adjust these before running ──────────────────────────────────────────────
SAMPLING_STRATEGY = "none"    # "none" | "udr" | "adr"
ENV_TYPE          = "source"  # "source" | "target"
TIMESTEPS         = 200_000   # total training steps
LOAD_MODEL        = False     # True → load existing zip instead of training
# ─────────────────────────────────────────────────────────────────────────────


## Train SAC on PandaPush-v3


In [ ]:

import os

save_name = os.path.join(
    REPO_ROOT, "part2", "models",
    f"sac_push_{SAMPLING_STRATEGY}_{ENV_TYPE}_{TIMESTEPS // 1000}k"
)
os.makedirs(os.path.dirname(save_name), exist_ok=True)

env = gym.make("PandaPush-v3", render_mode="rgb_array", type=ENV_TYPE, reward_type="dense")
if SAMPLING_STRATEGY != "none":
    env = RandomizationWrapper(env, mode=SAMPLING_STRATEGY)

if LOAD_MODEL:
    model = SAC.load(f"{save_name}.zip")
    model.set_env(DummyVecEnv([lambda: env]))
    print(f"Model loaded from {save_name}.zip")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training on device: {device}")

    vec_env = DummyVecEnv([lambda: env])

    model = SAC(
        policy="MultiInputPolicy",
        env=vec_env,
        device=device,
        verbose=1,
        learning_rate=1e-3,
        buffer_size=200_000,
        batch_size=256,
        tensorboard_log=f"{save_name}/logs",
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=50_000,
        save_path=f"{save_name}/checkpoints",
        name_prefix="model",
    )

    model.learn(total_timesteps=TIMESTEPS, callback=checkpoint_cb, progress_bar=True)
    model.save(save_name)
    print(f"Model saved to {save_name}.zip")


Created object with mass: 1.0
Training on device: cpu
Using cpu device
Logging to /content/FAIML-RL-26/part2/models/sac_push_none_source_200k/logs/SAC_1


Output()

---------------------------------
| rollout/           |          |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 42       |
|    time_elapsed    | 4        |
|    total_timesteps | 200      |
| train/             |          |
|    actor_loss      | -4.25    |
|    critic_loss     | 0.0717   |
|    ent_coef        | 0.907    |
|    ent_coef_loss   | -0.498   |
|    learning_rate   | 0.001    |
|    n_updates       | 99       |
---------------------------------
---------------------------------
| rollout/           |          |
|    success_rate    | 0.125    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 34       |
|    time_elapsed    | 10       |
|    total_timesteps | 351      |
| train/             |          |
|    actor_loss      | -4.64    |
|    critic_loss     | 0.0274   |
|    ent_coef        | 0.779    |
|    ent_coef_loss   | -1.25    |
|    learning_

## Evaluate the Trained Model


In [ ]:

N_EVAL_EPISODES = 20

eval_env = gym.make("PandaPush-v3", render_mode="rgb_array", type=ENV_TYPE, reward_type="dense")

episode_returns = []
successes = []

for ep in range(1, N_EVAL_EPISODES + 1):
    obs, info = eval_env.reset()
    done = False
    ep_return = 0.0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_return += float(reward)
        done = terminated or truncated

    episode_returns.append(ep_return)
    if isinstance(info, dict) and "is_success" in info:
        successes.append(float(info["is_success"]))
    print(f"Episode {ep:03d} | return = {ep_return:.3f}")

eval_env.close()

returns = np.array(episode_returns)
print(f"\n=== Eval over {N_EVAL_EPISODES} episodes ===")
print(f"Mean return : {returns.mean():.3f} ± {returns.std():.3f}")
print(f"Min / Max   : {returns.min():.3f} / {returns.max():.3f}")
if successes:
    print(f"Success rate: {np.mean(successes):.2%}")
